In [2]:
import os
os.environ["AI2POT_PATH"] = "/data/home/liuhanyu/mycode/AI2Pot/"
os.environ["AI2POT_TEST_DATA_PATH"] = os.path.join(os.environ.get("AI2POT_PATH"),
                                                   "test",
                                                   "test_data",
                                                   "POSCARs")
poscar_path: str = os.path.join(os.environ.get("AI2POT_TEST_DATA_PATH"), "POSCAR")

from typing import List
import torch
from pymatgen.core import Structure
from ai2pot.utils.usepot import (
    MlffInput,
    MlffToEFLossInput,
    MlffToLossInput)

In [3]:
type_map: List[int] = [16, 34, 41, 75]
rcut: float = 6.0
umax_num_neighs: int = 100
pbc_xyz: List[bool] = [True, True, True]
sort: bool = True
dtype: torch._C.dtype = torch.float32
device: torch._C.device = torch.device("cpu")

# 1. `ai2pot.utils.usepot`

## 1.1. predict ef/efv

In [4]:
structure: Structure = Structure.from_file(poscar_path)
mlff_input: MlffInput = MlffInput(type_map=type_map,
                                  rcut=rcut,
                                  umax_num_neighs=umax_num_neighs,
                                  pbc_xyz=pbc_xyz,
                                  sort=sort,
                                  dtype=dtype,
                                  device=device)
mlff_input_info = mlff_input.analyse_pymatgen(structure=structure)
print(mlff_input_info[0].shape)
print(mlff_input_info[1].shape)
print(mlff_input_info[2].shape)
print(mlff_input_info[3].shape)
print(mlff_input_info[4].shape)
print(mlff_input_info[5].shape)
print(mlff_input_info[6].shape)

torch.Size([1])
torch.Size([1, 108])
torch.Size([1, 108])
torch.Size([1, 108, 100])
torch.Size([1, 108, 100, 3])
torch.Size([1, 108])
torch.Size([1])


## 1.2. predict loss (contain energy, force, virial)

In [4]:
structure: Structure = Structure.from_file(poscar_path)
mlff_input: MlffToLossInput = MlffToLossInput(type_map=type_map,
                                              rcut=rcut,
                                              umax_num_neighs=umax_num_neighs,
                                              pbc_xyz=pbc_xyz,
                                              sort=sort,
                                              dtype=dtype,
                                              device=device)
mlff_input_info = mlff_input.analyse_pymatgen(structure=structure,
                                              e_weight=1.0,
                                              f_weight=1.0,
                                              v_weight=1.0)
print(mlff_input_info[0])
print(mlff_input_info[1])
print(mlff_input_info[2])
print(mlff_input_info[3].shape)
print(mlff_input_info[4].shape)
print(mlff_input_info[5].shape)

print(mlff_input_info[6].shape)
print(mlff_input_info[7].shape)
print(mlff_input_info[8].shape)
print(mlff_input_info[9].shape)
print(mlff_input_info[10].shape)
print(mlff_input_info[11].shape)
print(mlff_input_info[12].shape)

1.0
1.0
1.0
torch.Size([1])
torch.Size([1, 108, 3])
torch.Size([1, 3, 3])
torch.Size([1])
torch.Size([1, 108])
torch.Size([1, 108])
torch.Size([1, 108, 100])
torch.Size([1, 108, 100, 3])
torch.Size([1, 108])
torch.Size([1])


## 1.3. predict loss (contain energy, force)

In [5]:
structure: Structure = Structure.from_file(poscar_path)
mlff_input: MlffToEFLossInput = MlffToEFLossInput(type_map=type_map,
                                                  rcut=rcut,
                                                  umax_num_neighs=umax_num_neighs,
                                                  pbc_xyz=pbc_xyz,
                                                  sort=sort,
                                                  dtype=dtype,
                                                  device=device)
mlff_input_info = mlff_input.analyse_pymatgen(structure=structure,
                                              e_weight=1.0,
                                              f_weight=1.0)
print(mlff_input_info[0])
print(mlff_input_info[1])
print(mlff_input_info[2].shape)
print(mlff_input_info[3].shape)

print(mlff_input_info[4].shape)
print(mlff_input_info[5].shape)
print(mlff_input_info[6].shape)
print(mlff_input_info[7].shape)
print(mlff_input_info[8].shape)
print(mlff_input_info[9].shape)
print(mlff_input_info[10].shape)

1.0
1.0
torch.Size([1])
torch.Size([1, 108, 3])
torch.Size([1])
torch.Size([1, 108])
torch.Size([1, 108])
torch.Size([1, 108, 100])
torch.Size([1, 108, 100, 3])
torch.Size([1, 108])
torch.Size([1])


# 2. `ai2pot.utils.prepot`

In [6]:
from ai2pot.utils.prepot import ExtxyzShifter

## 2.1. `ai2pot.utils.prepot.ExtxyzShifter`

In [9]:
extxyz_path: str = os.path.join(os.environ.get("AI2POT_PATH"),
                                                "test",
                                                "test_data",
                                                "XYZ",
                                                "11_NEP_potential_PbTe",
                                                "train_m.xyz")
extxyz_shifter: ExtxyzShifter = ExtxyzShifter(extxyz_path=extxyz_path)

In [10]:
print("energy_shifts =", extxyz_shifter.types_energy_shifts)

energy_shifts = [-3.77207813 -3.77207813]


In [11]:
print("shifted_energies = ", extxyz_shifter.shifted_energies)

shifted_energies =  [ 5.82853200e+00 -1.74196800e+00  5.75253200e+00 -2.33376800e+00
  5.36193200e+00  6.06013200e+00  3.91332000e-01 -5.62296800e+00
  6.46673200e+00  3.93200000e-03 -3.00226800e+00  1.20784320e+01
  6.38463200e+00 -1.53372680e+01 -1.53447680e+01 -2.97168000e-01
 -5.61116800e+00  1.38438320e+01 -1.53393680e+01 -1.53385680e+01
 -3.60768000e-01  1.27079320e+01 -1.64516800e+00 -5.19036800e+00
  1.22856320e+01 -1.59872116e-13]
